# Projeto Computacional de Mecânica Clássica

In [37]:
from __future__ import annotations
from typing import Iterable
import math

In [38]:
def hello() -> None:
    print("Hello, world!")
    return

hello()

Hello, world!


## Calculo de Vetores

In [39]:
class Vec3:
    __slots__ = ("x", "y", "z")

    def __init__(self, x: float, y: float, z: float) -> None:
        self.x = x
        self.y = y
        self.z = z
    def __repr__(self) -> str:
        return f"Vec3({self.x}, {self.y}, {self.z})"

    def __add__(self, other: Vec3 | float) -> Vec3:
        if isinstance(other, Vec3):
            return Vec3(self.x + other.x, self.y + other.y, self.z + other.z)
        return Vec3(self.x + other, self.y + other, self.z + other)
    def __radd__(self, other: float) -> Vec3:
        return self + other
    def __iadd__(self, other: Vec3 | float) -> Vec3:
        if isinstance(other, Vec3):
            self.x += other.x
            self.y += other.y
            self.z += other.z
        else:
            self.x += other
            self.y += other
            self.z += other
        return self
    def __sub__(self, other: Vec3 | float) -> Vec3:
        if isinstance(other, Vec3):
            return Vec3(self.x - other.x, self.y - other.y, self.z - other.z)
        return Vec3(self.x - other, self.y - other, self.z - other)
    def __rsub__(self, other: float) -> Vec3:
        return Vec3(other - self.x, other - self.y, other - self.z)
    def __isub__(self, other: Vec3 | float) -> Vec3:
        if isinstance(other, Vec3):
            self.x -= other.x
            self.y -= other.y
            self.z -= other.z
        else:
            self.x -= other
            self.y -= other
            self.z -= other
        return self
    def __neg__(self) -> Vec3:
        return Vec3(-self.x, -self.y, -self.z)
    def __mul__(self, other: Vec3 | float) -> float | Vec3:
        if isinstance(other, Vec3):
            # DOT PRODUCT
            return self.x * other.x + self.y * other.y + self.z * other.z
        return Vec3(self.x * other, self.y * other, self.z * other)
    def __rmul__(self, other: float) -> Vec3:
        return self * other
    def __imul__(self, other: float) -> Vec3:
        self.x *= other
        self.y *= other
        self.z *= other
        return self
    def __truediv__(self, other: float) -> Vec3:
        return Vec3(self.x / other, self.y / other, self.z / other)
    def __itruediv__(self, other: float) -> Vec3:
        self.x /= other
        self.y /= other
        self.z /= other
        return self
    def dot(self, other: Vec3) -> float:
        return self * other
    def __matmul__(self, other: Vec3) -> float:
        return self * other
    def cross(self, other: Vec3) -> Vec3:
        return Vec3(
            self.y * other.z - self.z * other.y,
            self.z * other.x - self.x * other.z,
            self.x * other.y - self.y * other.x,
        )
    def length(self) -> float:
        return math.sqrt(self * self)
    def normalize(self) -> Vec3:
        l = self.length()
        if l == 0:
            raise ValueError("Cannot normalize zero vector")
        return self / l

def sum_vec3(vectors: Iterable[Vec3]) -> Vec3:
    result = Vec3(0.0, 0.0, 0.0)
    for v in vectors:
        result += v
    return result

## Conceito de uma particula

In [40]:
class Particle:
    def __init__(self, mass: float, position: Vec3, velocity: Vec3) -> None:
        self.mass = mass
        self.position = position
        self.velocity = velocity
    def __repr__(self) -> str:
        return f"Particle(mass={self.mass}, position={self.position}, velocity={self.velocity})"
    
    def update_position(self, dt: float) -> None:
        self.position += self.velocity * dt
    def update_velocity(self, forces: list[Vec3], dt: float) -> None:
        total_force = sum_vec3(forces)
        acceleration = total_force / self.mass
        self.velocity += acceleration * dt

## Forças

### Gravidade

In [41]:
GRAVITATIONAL_ACCELERATION = 9.81

def gravitational_force(particle: Particle) -> Vec3:
    return Vec3(0.0, 0.0, -particle.mass * GRAVITATIONAL_ACCELERATION)

### Força de arrasto


In [42]:
BETA_O = 1.0 
LAMBDA_O = 1.0
R_BETA = 1.0
R_LAMBDA = 1.0  
H = 1.0         # Escala característica de altura do meio

def drag_beta_up(particle: Particle, beta_o: float) -> float:
    return beta_o * math.exp(-particle.position.z / H)
def drag_lambda_up(particle: Particle, lambda_o: float) -> float:
    return lambda_o * math.exp(-particle.position.z / H)
def drag_beta_down(particle: Particle, beta_o: float, r_beta:float) -> float:
    return r_beta*drag_beta_up(particle, beta_o)
def drag_lambda_down(particle: Particle, lambda_o: float, r_lambda:float) -> float:
    return r_lambda*drag_lambda_up(particle, lambda_o)
def drag_force(particle: Particle) -> Vec3:
    velocity_z = particle.velocity.z
    if velocity_z > 0:
        beta = drag_beta_up(particle, BETA_O)
        lambda_ = drag_lambda_up(particle, LAMBDA_O)
        return Vec3(0.0, 0.0, 0.0)
    else:
        beta = drag_beta_down(particle, BETA_O, R_BETA)
        lambda_ = drag_lambda_down(particle, LAMBDA_O, R_LAMBDA)
        return Vec3(0.0, 0.0, 0.0)

## Simulação

In [43]:
TIME_STEP = 0.01

def main():
    particle = Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.0), velocity=Vec3(0.0, 0.0, 10.0))
    print(particle)
    for _ in range(10):
        forces = [gravitational_force(particle)]
        particle.update_velocity(forces, TIME_STEP)
        particle.update_position(TIME_STEP)
        print(particle)
main()

Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.0), velocity=Vec3(0.0, 0.0, 10.0))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.099019), velocity=Vec3(0.0, 0.0, 9.9019))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.19705699999999998), velocity=Vec3(0.0, 0.0, 9.803799999999999))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.294114), velocity=Vec3(0.0, 0.0, 9.705699999999998))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.39019), velocity=Vec3(0.0, 0.0, 9.607599999999998))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.48528499999999997), velocity=Vec3(0.0, 0.0, 9.509499999999997))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.579399), velocity=Vec3(0.0, 0.0, 9.411399999999997))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.6725319999999999), velocity=Vec3(0.0, 0.0, 9.313299999999996))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.7646839999999999), velocity=Vec3(0.0, 0.0, 9.215199999999996))
Particle(mass=1.0, position=Vec3(0.0, 0.0, 0.8558549999999999), velocity=Vec3(0.0, 0.0, 9.117099999999995)